# Лабораторная работа № 3
## Профилирование и качество данных: переход от raw к staging

**Проект:** Seoul Bike Sharing Demand  
**Группа:** ИВТ-241

### Аналитическая задача

Определить, в какие часы, сезоны и типы дней наблюдается наиболее высокий и низкий спрос на аренду велосипедов в Сеуле, а также исследовать связь спроса с погодными условиями.

### Цель работы

Построить профиль исходного набора данных, проверить качество значимых полей и подготовить нормализованные данные для дальнейшего анализа.

### Источник данных

Исходный CSV читается напрямую из S3-хранилища SeaweedFS. Объект слоя `raw` не изменяется.

In [1]:
from pathlib import Path
import os

import duckdb
from dotenv import load_dotenv

# Находим корень проекта и загружаем локальные настройки.
candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(
    (path for path in candidates if (path / ".env").is_file()),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Не найден .env в папке проекта.")

load_dotenv(PROJECT_ROOT / ".env")

# Параметры исходного объекта и результатов.
INGESTED_ON = "2026-10-13"

RAW_S3_PATH = (
    f"s3://raw/seoul_bike/"
    f"ingested_on={INGESTED_ON}/SeoulBikeData.csv"
)

OUTPUT_PREFIX = (
    f"s3://staging/seoul_bike/"
    f"source_ingested_on={INGESTED_ON}"
)

# Проверяем наличие настроек S3.
S3_ENDPOINT = os.getenv("S3_ENDPOINT", "localhost:8333")
S3_ACCESS_KEY = os.getenv("S3_ACCESS_KEY")
S3_SECRET_KEY = os.getenv("S3_SECRET_KEY")

if not S3_ACCESS_KEY or not S3_SECRET_KEY:
    raise RuntimeError(
        "В .env должны быть заданы S3_ACCESS_KEY и S3_SECRET_KEY."
    )

# Подключаем DuckDB к SeaweedFS.
con = duckdb.connect()

con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")

s3_settings = {
    "s3_endpoint": S3_ENDPOINT,
    "s3_access_key_id": S3_ACCESS_KEY,
    "s3_secret_access_key": S3_SECRET_KEY,
    "s3_region": "us-east-1",
}

for name, value in s3_settings.items():
    escaped_value = value.replace("'", "''")
    con.execute(f"SET {name} = '{escaped_value}';")

con.execute("SET s3_use_ssl = false;")
con.execute("SET s3_url_style = 'path';")

# Описываем источник. Hive-партиция добавляет ingested_on.
escaped_path = RAW_S3_PATH.replace("'", "''")

RAW_RELATION = (
    f"read_csv_auto("
    f"'{escaped_path}', "
    f"encoding='latin-1', "
    f"hive_partitioning=true"
    f")"
)

print("Корень проекта:", PROJECT_ROOT)
print("Исходный объект:", RAW_S3_PATH)
print("Префикс результатов:", OUTPUT_PREFIX)
print("DuckDB:", duckdb.__version__)

Корень проекта: c:\Users\Даниил\Desktop\Учусь типа\Семестр 5\ИТАД\SeoulBikeLab2
Исходный объект: s3://raw/seoul_bike/ingested_on=2026-10-13/SeoulBikeData.csv
Префикс результатов: s3://staging/seoul_bike/source_ingested_on=2026-10-13
DuckDB: 1.2.0


In [2]:
# Общие характеристики исходного набора.
summary = con.execute(
    f"""
    SELECT
        COUNT(*) AS rows_count,
        COUNT(DISTINCT Date) AS dates_count,
        MIN(Date) AS min_date,
        MAX(Date) AS max_date
    FROM {RAW_RELATION}
    """
).fetchone()

print("Количество строк:", summary[0])
print("Количество уникальных дат:", summary[1])
print("Минимальная дата:", summary[2])
print("Максимальная дата:", summary[3])

# Фактическая схема данных.
print("\nСхема исходного набора:")
schema = con.execute(
    f"DESCRIBE SELECT * FROM {RAW_RELATION}"
).fetchall()

for column in schema:
    print(f"{column[0]} — {column[1]}")

# Несколько строк для визуальной проверки.
sample_query = con.execute(
    f"SELECT * FROM {RAW_RELATION} LIMIT 5"
)

sample_columns = [item[0] for item in sample_query.description]
sample_rows = sample_query.fetchall()

print("\nПервые пять строк:")
print(" | ".join(sample_columns))

for row in sample_rows:
    print(" | ".join(str(value) for value in row))

Количество строк: 8760
Количество уникальных дат: 365
Минимальная дата: 2017-12-01
Максимальная дата: 2018-11-30

Схема исходного набора:
Date — DATE
Rented Bike Count — BIGINT
Hour — BIGINT
Temperature(°C) — DOUBLE
Humidity(%) — BIGINT
Wind speed (m/s) — DOUBLE
Visibility (10m) — BIGINT
Dew point temperature(°C) — DOUBLE
Solar Radiation (MJ/m2) — DOUBLE
Rainfall(mm) — DOUBLE
Snowfall (cm) — DOUBLE
Seasons — VARCHAR
Holiday — VARCHAR
Functioning Day — BOOLEAN
ingested_on — DATE

Первые пять строк:
Date | Rented Bike Count | Hour | Temperature(°C) | Humidity(%) | Wind speed (m/s) | Visibility (10m) | Dew point temperature(°C) | Solar Radiation (MJ/m2) | Rainfall(mm) | Snowfall (cm) | Seasons | Holiday | Functioning Day | ingested_on
2017-12-01 | 254 | 0 | -5.2 | 37 | 2.2 | 2000 | -17.6 | 0.0 | 0.0 | 0.0 | Winter | No Holiday | True | 2026-10-13
2017-12-01 | 204 | 1 | -5.5 | 38 | 0.8 | 2000 | -17.6 | 0.0 | 0.0 | 0.0 | Winter | No Holiday | True | 2026-10-13
2017-12-01 | 173 | 2 | -6.0 | 

## Профиль данных

### Полнота значимых полей

Проверяем наличие пропущенных значений в полях, необходимых для анализа почасового спроса на аренду велосипедов.

In [3]:
# Создаём временное представление исходного CSV.
# Оно используется только для чтения и не изменяет объект в raw.
con.execute(
    f"""
    CREATE OR REPLACE TEMP VIEW raw_data AS
    SELECT * FROM {RAW_RELATION}
    """
)

# Профилируем полноту пяти значимых полей.
selected_fields = [
    ("Date", '"Date"'),
    ("Hour", '"Hour"'),
    ("Rented Bike Count", '"Rented Bike Count"'),
    ("Humidity(%)", '"Humidity(%)"'),
    ("Functioning Day", '"Functioning Day"'),
]

total_rows = con.execute(
    "SELECT COUNT(*) FROM raw_data"
).fetchone()[0]

print(f"Всего проверено строк: {total_rows}\n")
print("Поле | Пропуски | Доля пропусков, %")

missing_profile = []

for field_name, column_sql in selected_fields:
    missing_count = con.execute(
        f"""
        SELECT COUNT(*)
        FROM raw_data
        WHERE {column_sql} IS NULL
        """
    ).fetchone()[0]

    missing_share = (
        missing_count / total_rows * 100
        if total_rows > 0
        else 0
    )

    missing_profile.append(
        (field_name, missing_count, missing_share)
    )

    print(
        f"{field_name} | {missing_count} | "
        f"{missing_share:.4f}"
    )

Всего проверено строк: 8760

Поле | Пропуски | Доля пропусков, %
Date | 0 | 0.0000
Hour | 0 | 0.0000
Rented Bike Count | 0 | 0.0000
Humidity(%) | 0 | 0.0000
Functioning Day | 0 | 0.0000


### Уникальность почасовых наблюдений

Бизнес-ключ — комбинация `Date + Hour`. Каждая строка должна соответствовать одному часу определённой даты. Повторения ключа могут привести к двойному учёту аренд при анализе спроса по часам, сезонам и типам дней.

Дополнительно проверяется наличие полностью совпадающих строк.

In [4]:
# Проверка повторов бизнес-ключа Date + Hour.
duplicate_keys = con.execute(
    """
    WITH key_counts AS (
        SELECT
            "Date",
            "Hour",
            COUNT(*) AS n
        FROM raw_data
        GROUP BY "Date", "Hour"
        HAVING COUNT(*) > 1
    )
    SELECT
        COUNT(*) AS duplicate_key_groups,
        COALESCE(SUM(n - 1), 0) AS excess_rows
    FROM key_counts
    """
).fetchone()

# Проверка полностью совпадающих строк.
full_duplicates = con.execute(
    """
    WITH row_counts AS (
        SELECT
            *,
            COUNT(*) AS n
        FROM raw_data
        GROUP BY ALL
        HAVING COUNT(*) > 1
    )
    SELECT COALESCE(SUM(n - 1), 0)
    FROM row_counts
    """
).fetchone()[0]

print("Групп повторяющегося ключа Date + Hour:", duplicate_keys[0])
print("Избыточных строк по ключу:", duplicate_keys[1])
print("Избыточных полностью совпадающих строк:", full_duplicates)

Групп повторяющегося ключа Date + Hour: 0
Избыточных строк по ключу: 0
Избыточных полностью совпадающих строк: 0


### Допустимость значений и согласованность полей

Проверяем диапазоны часа наблюдения, количества аренд и влажности воздуха. Дополнительно проверяем допустимые категории сезона и согласованность количества аренд с признаком функционирования системы.

Отрицательные значения температуры воздуха и точки росы не считаются ошибками, поскольку соответствуют физическому смыслу этих показателей.

In [5]:
# Проверка числовых диапазонов и согласованности полей.
range_profile = con.execute(
    """
    SELECT
        MIN("Hour") AS min_hour,
        MAX("Hour") AS max_hour,
        COUNT(*) FILTER (
            WHERE "Hour" < 0 OR "Hour" > 23
        ) AS invalid_hour_rows,

        MIN("Rented Bike Count") AS min_rentals,
        MAX("Rented Bike Count") AS max_rentals,
        COUNT(*) FILTER (
            WHERE "Rented Bike Count" < 0
        ) AS negative_rental_rows,

        MIN("Humidity(%)") AS min_humidity,
        MAX("Humidity(%)") AS max_humidity,
        COUNT(*) FILTER (
            WHERE "Humidity(%)" < 0 OR "Humidity(%)" > 100
        ) AS invalid_humidity_rows,

        COUNT(*) FILTER (
            WHERE "Seasons" NOT IN (
                'Winter', 'Spring', 'Summer', 'Autumn'
            )
        ) AS invalid_season_rows,

        COUNT(*) FILTER (
            WHERE "Functioning Day" = false
              AND "Rented Bike Count" <> 0
        ) AS closed_system_with_rentals
    FROM raw_data
    """
).fetchone()

labels = [
    "Минимальный час",
    "Максимальный час",
    "Строк с недопустимым часом",
    "Минимальное количество аренд",
    "Максимальное количество аренд",
    "Строк с отрицательным количеством аренд",
    "Минимальная влажность",
    "Максимальная влажность",
    "Строк с недопустимой влажностью",
    "Строк с неизвестной категорией сезона",
    "Строк: система не работала, но аренды есть",
]

for label, value in zip(labels, range_profile):
    print(f"{label}: {value}")

Минимальный час: 0
Максимальный час: 23
Строк с недопустимым часом: 0
Минимальное количество аренд: 0
Максимальное количество аренд: 3556
Строк с отрицательным количеством аренд: 0
Минимальная влажность: 0
Максимальная влажность: 98
Строк с недопустимой влажностью: 0
Строк с неизвестной категорией сезона: 0
Строк: система не работала, но аренды есть: 0


### Распределение наблюдений и спроса по часам

Для каждого часа суток рассчитывается количество наблюдений и среднее количество арендованных велосипедов. Распределение позволяет проверить полноту временного покрытия и подготовить данные для последующего анализа спроса по времени суток.

In [6]:
hourly_profile = con.execute(
    """
    SELECT
        "Hour" AS hour,
        COUNT(*) AS observations,
        AVG("Rented Bike Count") AS average_rentals,
        MIN("Rented Bike Count") AS min_rentals,
        MAX("Rented Bike Count") AS max_rentals
    FROM raw_data
    GROUP BY "Hour"
    ORDER BY "Hour"
    """
).fetchall()

print("Час | Наблюдений | Среднее число аренд | Мин. | Макс.")

for hour, count, average, minimum, maximum in hourly_profile:
    print(
        f"{hour:>2} | {count:>11} | {average:>20.2f} | "
        f"{minimum:>4} | {maximum:>4}"
    )

Час | Наблюдений | Среднее число аренд | Мин. | Макс.
 0 |         365 |               541.46 |    0 | 1394
 1 |         365 |               426.18 |    0 | 1088
 2 |         365 |               301.63 |    0 | 1254
 3 |         365 |               203.33 |    0 |  644
 4 |         365 |               132.59 |    0 |  421
 5 |         365 |               139.08 |    0 |  383
 6 |         365 |               287.56 |    0 |  807
 7 |         365 |               606.01 |    0 | 1629
 8 |         365 |              1015.70 |    0 | 2495
 9 |         365 |               645.98 |    0 | 1401
10 |         365 |               527.82 |    0 | 1269
11 |         365 |               600.85 |    0 | 1478
12 |         365 |               699.44 |    0 | 1798
13 |         365 |               733.25 |    0 | 2000
14 |         365 |               758.82 |    0 | 2128
15 |         365 |               829.19 |    0 | 2329
16 |         365 |               930.62 |    0 | 2479
17 |         365 |          

## Правила качества данных

Для каждого правила отдельно рассчитываются число проверенных строк, число нарушений, доля нарушений и превышение установленного порога.

Порог задан в количестве строк. Для всех правил допустимое число нарушений равно нулю. При обнаружении нескольких нарушений критические нарушения имеют приоритет: запись исключается из staging и помещается в карантин один раз с сохранением всех причин.

In [7]:
# Общие параметры и накопитель результатов проверок.
RULE_THRESHOLDS = {}
rule_results = []

### Правило Q01 — уникальность `Date + Hour`

**ID:** `unique_date_hour`

**Проверяемые строки:** все записи исходного набора.

**Условие:** комбинация `Date + Hour` должна встречаться ровно один раз, поскольку одна строка соответствует одному часу конкретной даты.

**Порог:** не более 0 нарушающих строк.

**Критичность:** `critical`.

**Действие:** `quarantine`. Все записи с повторяющимся ключом исключаются из staging, чтобы не допустить двойного учёта спроса.

**Связь с задачей:** повторы искажают количество наблюдений и среднее число аренд по часам, сезонам и типам дней.

In [8]:
# Q01: проверяем уникальность бизнес-ключа Date + Hour.
RULE_THRESHOLDS["unique_date_hour"] = 0
threshold = RULE_THRESHOLDS["unique_date_hour"]

unique_key_check = con.execute(
    """
    WITH keyed AS (
        SELECT
            "Date",
            "Hour",
            COUNT(*) OVER (
                PARTITION BY "Date", "Hour"
            ) AS key_count
        FROM raw_data
    )
    SELECT
        COUNT(*) AS checked_rows,
        COUNT(*) FILTER (
            WHERE key_count > 1
        ) AS violating_rows
    FROM keyed
    """
).fetchone()

checked_rows = unique_key_check[0]
violating_rows = unique_key_check[1]
violation_share = (
    violating_rows / checked_rows
    if checked_rows else 0
)

rule_results.append({
    "id": "unique_date_hour",
    "checked_rows": checked_rows,
    "violating_rows": violating_rows,
    "violation_share": violation_share,
    "threshold": threshold,
    "threshold_exceeded": violating_rows > threshold,
    "criticality": "critical",
    "action": "quarantine",
})

print("Правило: unique_date_hour")
print("Проверено строк:", checked_rows)
print("Нарушающих строк:", violating_rows)
print(f"Доля нарушений: {violation_share:.4%}")
print("Допустимый порог:", threshold)
print("Порог превышен:", violating_rows > threshold)

Правило: unique_date_hour
Проверено строк: 8760
Нарушающих строк: 0
Доля нарушений: 0.0000%
Допустимый порог: 0
Порог превышен: False


### Правило Q02 — полнота количества аренд

**ID:** `required_rented_bike_count`

**Проверяемые строки:** все записи исходного набора.

**Условие:** поле `Rented Bike Count` не должно содержать пропусков, поскольку используется как основная метрика спроса.

**Порог:** 0 пропусков.

**Критичность:** `critical`.

**Действие:** `quarantine`. Записи без значения целевой метрики исключаются из staging, поскольку их нельзя использовать для расчёта показателей спроса.

**Связь с задачей:** отсутствие количества аренд искажает число пригодных наблюдений и расчёты среднего, медианного, минимального и максимального спроса.

In [9]:
# Q02: проверяем полноту целевой метрики.
RULE_THRESHOLDS["required_rented_bike_count"] = 0
threshold = RULE_THRESHOLDS["required_rented_bike_count"]

completeness_check = con.execute(
    """
    SELECT
        COUNT(*) AS checked_rows,
        COUNT(*) FILTER (
            WHERE "Rented Bike Count" IS NULL
        ) AS violating_rows
    FROM raw_data
    """
).fetchone()

checked_rows = completeness_check[0]
violating_rows = completeness_check[1]
violation_share = (
    violating_rows / checked_rows
    if checked_rows else 0
)

rule_results.append({
    "id": "required_rented_bike_count",
    "checked_rows": checked_rows,
    "violating_rows": violating_rows,
    "violation_share": violation_share,
    "threshold": threshold,
    "threshold_exceeded": violating_rows > threshold,
    "criticality": "critical",
    "action": "quarantine",
})

print("Правило: required_rented_bike_count")
print("Проверено строк:", checked_rows)
print("Пропусков:", violating_rows)
print(f"Доля пропусков: {violation_share:.4%}")
print("Допустимый порог:", threshold)
print("Порог превышен:", violating_rows > threshold)

Правило: required_rented_bike_count
Проверено строк: 8760
Пропусков: 0
Доля пропусков: 0.0000%
Допустимый порог: 0
Порог превышен: False


### Правило Q03 — допустимый диапазон часа

**ID:** `hour_in_valid_range`

**Проверяемые строки:** все записи исходного набора.

**Условие:** поле `Hour` должно содержать значение от 0 до 23 включительно.

**Порог:** 0 нарушений.

**Критичность:** `critical`.

**Действие:** `quarantine`. Записи с отсутствующим или недопустимым часом исключаются из staging, поскольку их нельзя однозначно отнести к часу суток.

**Связь с задачей:** недопустимые значения часа искажают распределение спроса по времени суток.

In [10]:
# Q03: проверяем допустимый диапазон часа.
rule_id = "hour_in_valid_range"
RULE_THRESHOLDS[rule_id] = 0
threshold = RULE_THRESHOLDS[rule_id]

hour_check = con.execute(
    """
    SELECT
        COUNT(*) AS checked_rows,
        COUNT(*) FILTER (
            WHERE "Hour" IS NULL
               OR "Hour" < 0
               OR "Hour" > 23
        ) AS violating_rows
    FROM raw_data
    """
).fetchone()

checked_rows = hour_check[0]
violating_rows = hour_check[1]
violation_share = (
    violating_rows / checked_rows
    if checked_rows else 0
)

# Заменяем прежний результат этого правила при повторном запуске ячейки.
rule_results = [
    result for result in rule_results
    if result["id"] != rule_id
]

rule_results.append({
    "id": rule_id,
    "checked_rows": checked_rows,
    "violating_rows": violating_rows,
    "violation_share": violation_share,
    "threshold": threshold,
    "threshold_exceeded": violating_rows > threshold,
    "criticality": "critical",
    "action": "quarantine",
})

print("Правило:", rule_id)
print("Проверено строк:", checked_rows)
print("Нарушающих строк:", violating_rows)
print(f"Доля нарушений: {violation_share:.4%}")
print("Допустимый порог:", threshold)
print("Порог превышен:", violating_rows > threshold)

Правило: hour_in_valid_range
Проверено строк: 8760
Нарушающих строк: 0
Доля нарушений: 0.0000%
Допустимый порог: 0
Порог превышен: False


### Правило Q04 — неотрицательное количество аренд

**ID:** `non_negative_rented_bike_count`

**Проверяемые строки:** все записи исходного набора.

**Условие:** `Rented Bike Count` должно быть неотрицательным числом. Значение 0 допустимо, поскольку оно может соответствовать периоду, когда система не функционировала.

**Порог:** 0 нарушений.

**Критичность:** `critical`.

**Действие:** `quarantine`. Записи с отрицательным количеством аренд исключаются из staging, поскольку такие значения не имеют допустимого смысла для количества арендованных велосипедов.

**Связь с задачей:** отрицательные значения исказили бы средний, медианный, минимальный и суммарный спрос.

In [11]:
# Q04: проверяем неотрицательность количества аренд.
rule_id = "non_negative_rented_bike_count"
RULE_THRESHOLDS[rule_id] = 0
threshold = RULE_THRESHOLDS[rule_id]

rental_check = con.execute(
    """
    SELECT
        COUNT(*) AS checked_rows,
        COUNT(*) FILTER (
            WHERE "Rented Bike Count" < 0
        ) AS violating_rows
    FROM raw_data
    """
).fetchone()

checked_rows = rental_check[0]
violating_rows = rental_check[1]
violation_share = (
    violating_rows / checked_rows
    if checked_rows else 0
)

# Не допускаем дублирования результата при повторном запуске ячейки.
rule_results = [
    result for result in rule_results
    if result["id"] != rule_id
]

rule_results.append({
    "id": rule_id,
    "checked_rows": checked_rows,
    "violating_rows": violating_rows,
    "violation_share": violation_share,
    "threshold": threshold,
    "threshold_exceeded": violating_rows > threshold,
    "criticality": "critical",
    "action": "quarantine",
})

print("Правило:", rule_id)
print("Проверено строк:", checked_rows)
print("Нарушающих строк:", violating_rows)
print(f"Доля нарушений: {violation_share:.4%}")
print("Допустимый порог:", threshold)
print("Порог превышен:", violating_rows > threshold)

Правило: non_negative_rented_bike_count
Проверено строк: 8760
Нарушающих строк: 0
Доля нарушений: 0.0000%
Допустимый порог: 0
Порог превышен: False


### Правило Q05 — допустимый диапазон влажности

**ID:** `humidity_in_valid_range`

**Проверяемые строки:** все записи исходного набора.

**Условие:** `Humidity(%)` должна быть заполнена и находиться в диапазоне от 0 до 100 включительно.

**Порог:** 0 нарушений.

**Критичность:** `warning`.

**Действие:** `flag`. Запись остаётся в staging, но получает признак нарушения качества. Это позволяет сохранить наблюдение для анализа спроса, не скрывая проблему в погодных данных.

**Связь с задачей:** недопустимая влажность может исказить анализ связи погодных условий с количеством аренд.

In [12]:
# Q05: проверяем влажность в диапазоне 0–100%.
rule_id = "humidity_in_valid_range"
RULE_THRESHOLDS[rule_id] = 0
threshold = RULE_THRESHOLDS[rule_id]

humidity_check = con.execute(
    """
    SELECT
        COUNT(*) AS checked_rows,
        COUNT(*) FILTER (
            WHERE "Humidity(%)" IS NULL
               OR "Humidity(%)" < 0
               OR "Humidity(%)" > 100
        ) AS violating_rows
    FROM raw_data
    """
).fetchone()

checked_rows = humidity_check[0]
violating_rows = humidity_check[1]
violation_share = (
    violating_rows / checked_rows
    if checked_rows else 0
)

# При повторном запуске ячейки не дублируем правило.
rule_results = [
    result for result in rule_results
    if result["id"] != rule_id
]

rule_results.append({
    "id": rule_id,
    "checked_rows": checked_rows,
    "violating_rows": violating_rows,
    "violation_share": violation_share,
    "threshold": threshold,
    "threshold_exceeded": violating_rows > threshold,
    "criticality": "warning",
    "action": "flag",
})

print("Правило:", rule_id)
print("Проверено строк:", checked_rows)
print("Нарушающих строк:", violating_rows)
print(f"Доля нарушений: {violation_share:.4%}")
print("Допустимый порог:", threshold)
print("Порог превышен:", violating_rows > threshold)

Правило: humidity_in_valid_range
Проверено строк: 8760
Нарушающих строк: 0
Доля нарушений: 0.0000%
Допустимый порог: 0
Порог превышен: False


### Правило Q06 — согласованность работы системы и аренд

**ID:** `closed_system_zero_rentals`

**Проверяемые строки:** все записи исходного набора.

**Условие:** если `Functioning Day = false`, то `Rented Bike Count` должен быть равен 0.

**Порог:** 0 нарушений.

**Критичность:** `critical`.

**Действие:** `quarantine`. Записи с положительным количеством аренд при неработающей системе исключаются из staging, поскольку противоречат признаку функционирования системы.

**Связь с задачей:** несогласованные записи могут исказить сравнение спроса по часам, сезонам и типам дней.

In [13]:
# Q06: проверяем согласованность функционирования системы и аренд.
rule_id = "closed_system_zero_rentals"
RULE_THRESHOLDS[rule_id] = 0
threshold = RULE_THRESHOLDS[rule_id]

consistency_check = con.execute(
    """
    SELECT
        COUNT(*) AS checked_rows,
        COUNT(*) FILTER (
            WHERE "Functioning Day" = false
              AND "Rented Bike Count" <> 0
        ) AS violating_rows
    FROM raw_data
    """
).fetchone()

checked_rows = consistency_check[0]
violating_rows = consistency_check[1]
violation_share = (
    violating_rows / checked_rows
    if checked_rows else 0
)

# Защита от дублирования при повторном выполнении ячейки.
rule_results = [
    result for result in rule_results
    if result["id"] != rule_id
]

rule_results.append({
    "id": rule_id,
    "checked_rows": checked_rows,
    "violating_rows": violating_rows,
    "violation_share": violation_share,
    "threshold": threshold,
    "threshold_exceeded": violating_rows > threshold,
    "criticality": "critical",
    "action": "quarantine",
})

print("Правило:", rule_id)
print("Проверено строк:", checked_rows)
print("Нарушающих строк:", violating_rows)
print(f"Доля нарушений: {violation_share:.4%}")
print("Допустимый порог:", threshold)
print("Порог превышен:", violating_rows > threshold)

Правило: closed_system_zero_rentals
Проверено строк: 8760
Нарушающих строк: 0
Доля нарушений: 0.0000%
Допустимый порог: 0
Порог превышен: False


## Сводные результаты проверки качества

Ниже приведены результаты всех правил качества, включая число проверенных строк, число и долю нарушений, установленный порог, критичность и действие по проблемным записям.

In [14]:
from IPython.display import display, Markdown

# Ожидаемый порядок правил в сводной таблице.
rule_order = [
    "unique_date_hour",
    "required_rented_bike_count",
    "hour_in_valid_range",
    "non_negative_rented_bike_count",
    "humidity_in_valid_range",
    "closed_system_zero_rentals",
]

# Получаем последние результаты каждого правила.
results_by_id = {result["id"]: result for result in rule_results}

missing_rules = [
    rule_id for rule_id in rule_order
    if rule_id not in results_by_id
]

if missing_rules:
    raise ValueError(
        "Нет результатов правил: "
        + ", ".join(missing_rules)
        + ". Выполни соответствующие ячейки правил заново."
    )

# Формируем Markdown-таблицу без зависимости от pandas и NumPy.
table = (
    "| ID правила | Проверено строк | Нарушений | "
    "Доля нарушений | Порог, строк | Порог превышен | "
    "Критичность | Действие |\n"
    "|---|---:|---:|---:|---:|---|---|---|\n"
)

for rule_id in rule_order:
    result = results_by_id[rule_id]

    table += (
        f"| `{result['id']}` "
        f"| {result['checked_rows']} "
        f"| {result['violating_rows']} "
        f"| {result['violation_share']:.4%} "
        f"| {result['threshold']} "
        f"| {'Да' if result['threshold_exceeded'] else 'Нет'} "
        f"| {result['criticality']} "
        f"| `{result['action']}` |\n"
    )

display(Markdown(table))

| ID правила | Проверено строк | Нарушений | Доля нарушений | Порог, строк | Порог превышен | Критичность | Действие |
|---|---:|---:|---:|---:|---|---|---|
| `unique_date_hour` | 8760 | 0 | 0.0000% | 0 | Нет | critical | `quarantine` |
| `required_rented_bike_count` | 8760 | 0 | 0.0000% | 0 | Нет | critical | `quarantine` |
| `hour_in_valid_range` | 8760 | 0 | 0.0000% | 0 | Нет | critical | `quarantine` |
| `non_negative_rented_bike_count` | 8760 | 0 | 0.0000% | 0 | Нет | critical | `quarantine` |
| `humidity_in_valid_range` | 8760 | 0 | 0.0000% | 0 | Нет | warning | `flag` |
| `closed_system_zero_rentals` | 8760 | 0 | 0.0000% | 0 | Нет | critical | `quarantine` |


## Подготовка staging и карантина

Для каждой исходной записи повторно вычисляются признаки нарушения правил качества. Записи с нарушением хотя бы одного критического правила направляются в карантин. Остальные записи сохраняются в staging. Нарушения предупреждающего правила влажности не исключают запись, а фиксируются отдельным признаком.

Исходный объект raw остаётся неизменным.

In [15]:
# Повторно оцениваем правила на уровне каждой исходной записи.
# Временное представление создаётся только в текущем соединении DuckDB.

con.execute("""
CREATE OR REPLACE TEMP VIEW quality_evaluated AS
WITH keyed AS (
    SELECT
        *,
        COUNT(*) OVER (
            PARTITION BY "Date", "Hour"
        ) AS date_hour_count
    FROM raw_data
)
SELECT
    * EXCLUDE (date_hour_count),

    COALESCE(date_hour_count > 1, FALSE)
        AS q_unique_date_hour,

    ("Rented Bike Count" IS NULL)
        AS q_required_rented_bike_count,

    (
        "Hour" IS NULL
        OR "Hour" < 0
        OR "Hour" > 23
    ) AS q_hour_in_valid_range,

    COALESCE("Rented Bike Count" < 0, FALSE)
        AS q_non_negative_rented_bike_count,

    (
        "Humidity(%)" IS NULL
        OR "Humidity(%)" < 0
        OR "Humidity(%)" > 100
    ) AS q_humidity_in_valid_range,

    COALESCE(
        "Functioning Day" = FALSE
        AND "Rented Bike Count" <> 0,
        FALSE
    ) AS q_closed_system_zero_rentals

FROM keyed
""")

# Критические нарушения исключают запись из staging.
critical_condition = """
    COALESCE(q_unique_date_hour, FALSE)
    OR COALESCE(q_required_rented_bike_count, FALSE)
    OR COALESCE(q_hour_in_valid_range, FALSE)
    OR COALESCE(q_non_negative_rented_bike_count, FALSE)
    OR COALESCE(q_closed_system_zero_rentals, FALSE)
"""

quality_totals = con.execute(
    f"""
    SELECT
        COUNT(*) AS raw_rows,

        COUNT(*) FILTER (
            WHERE NOT ({critical_condition})
        ) AS staging_rows,

        COUNT(*) FILTER (
            WHERE ({critical_condition})
        ) AS quarantine_rows,

        COUNT(*) FILTER (
            WHERE q_humidity_in_valid_range
                IS DISTINCT FROM (
                    "Humidity(%)" IS NULL
                    OR "Humidity(%)" < 0
                    OR "Humidity(%)" > 100
                )
              AND NOT ({critical_condition})
        ) AS humidity_flag_mismatch_rows

    FROM quality_evaluated
    """
).fetchone()

raw_rows, staging_rows, quarantine_rows, humidity_flag_mismatch_rows = quality_totals

print("Исходных строк:", raw_rows)
print("Строк для staging:", staging_rows)
print("Строк для карантина:", quarantine_rows)
print("Несоответствий флага влажности:", humidity_flag_mismatch_rows)

print(
    "Баланс raw = staging + карантин:",
    raw_rows == staging_rows + quarantine_rows
)

assert raw_rows == staging_rows + quarantine_rows, (
    "Баланс записей не выполняется!"
)

assert humidity_flag_mismatch_rows == 0, (
    "Флаг влажности не соответствует фактическому значению."
)


Исходных строк: 8760
Строк для staging: 8760
Строк для карантина: 0
Несоответствий флага влажности: 0
Баланс raw = staging + карантин: True


### Нормализация и сохранение результатов

Имена аналитических полей приводятся к формату snake_case. Критические нарушения направляются в карантин, а записи без критических нарушений сохраняются в staging. Для предупреждающего правила влажности сохраняется отдельный булев признак.

Исходный объект raw не изменяется.

In [16]:
# Подготавливаем итоговое условие критических нарушений.
critical_condition = """
    COALESCE(q_unique_date_hour, FALSE)
    OR COALESCE(q_required_rented_bike_count, FALSE)
    OR COALESCE(q_hour_in_valid_range, FALSE)
    OR COALESCE(q_non_negative_rented_bike_count, FALSE)
    OR COALESCE(q_closed_system_zero_rentals, FALSE)
"""

# Нормализуем названия полей и сохраняем предупреждающий флаг.
con.execute(f"""
CREATE OR REPLACE TEMP VIEW staging_data AS
SELECT
    "Date" AS observation_date,
    "Rented Bike Count" AS rented_bike_count,
    "Hour" AS hour,
    "Temperature(°C)" AS temperature_c,
    "Humidity(%)" AS humidity_pct,
    "Wind speed (m/s)" AS wind_speed_m_s,
    "Visibility (10m)" AS visibility_10m,
    "Dew point temperature(°C)" AS dew_point_temperature_c,
    "Solar Radiation (MJ/m2)" AS solar_radiation_mj_m2,
    "Rainfall(mm)" AS rainfall_mm,
    "Snowfall (cm)" AS snowfall_cm,
    "Seasons" AS season,
    "Holiday" AS holiday,
    "Functioning Day" AS functioning_day,
    ingested_on,
    q_humidity_in_valid_range
        AS quality_flag_humidity_out_of_range
FROM quality_evaluated
WHERE NOT ({critical_condition})
""")

# Карантин хранит исходные поля, все критические причины
# и идентификаторы правил. Каждая исходная запись попадёт туда один раз.
con.execute(f"""
CREATE OR REPLACE TEMP VIEW quarantine_data AS
SELECT
    "Date",
    "Rented Bike Count",
    "Hour",
    "Temperature(°C)",
    "Humidity(%)",
    "Wind speed (m/s)",
    "Visibility (10m)",
    "Dew point temperature(°C)",
    "Solar Radiation (MJ/m2)",
    "Rainfall(mm)",
    "Snowfall (cm)",
    "Seasons",
    "Holiday",
    "Functioning Day",
    ingested_on,

    concat_ws('; ',
        CASE WHEN q_unique_date_hour
            THEN 'unique_date_hour' END,
        CASE WHEN q_required_rented_bike_count
            THEN 'required_rented_bike_count' END,
        CASE WHEN q_hour_in_valid_range
            THEN 'hour_in_valid_range' END,
        CASE WHEN q_non_negative_rented_bike_count
            THEN 'non_negative_rented_bike_count' END,
        CASE WHEN q_closed_system_zero_rentals
            THEN 'closed_system_zero_rentals' END
    ) AS quality_rule_ids,

    concat_ws('; ',
        CASE WHEN q_unique_date_hour
            THEN 'Повторяющийся ключ Date + Hour' END,
        CASE WHEN q_required_rented_bike_count
            THEN 'Отсутствует количество аренд' END,
        CASE WHEN q_hour_in_valid_range
            THEN 'Час вне диапазона 0–23 или отсутствует' END,
        CASE WHEN q_non_negative_rented_bike_count
            THEN 'Отрицательное количество аренд' END,
        CASE WHEN q_closed_system_zero_rentals
            THEN 'Количество аренд при неработающей системе' END
    ) AS quarantine_reason

FROM quality_evaluated
WHERE ({critical_condition})
""")

# Пути берём из параметра OUTPUT_PREFIX.
staging_path = f"{OUTPUT_PREFIX}/staging.parquet"
quarantine_path = f"{OUTPUT_PREFIX}/quarantine.parquet"

# Пересчитываем фактические количества перед записью.
raw_count = con.execute(
    "SELECT COUNT(*) FROM raw_data"
).fetchone()[0]

staging_count = con.execute(
    "SELECT COUNT(*) FROM staging_data"
).fetchone()[0]

quarantine_count = con.execute(
    "SELECT COUNT(*) FROM quarantine_data"
).fetchone()[0]

assert raw_count == staging_count + quarantine_count, (
    "Баланс не выполняется: raw должен равняться staging + quarantine."
)

# Записываем оба результата в S3. Опция позволяет повторять запись
# в тот же объект без ошибки из-за его существования.
escaped_staging_path = staging_path.replace("'", "''")
escaped_quarantine_path = quarantine_path.replace("'", "''")

con.execute(f"""
COPY (SELECT * FROM staging_data)
TO '{escaped_staging_path}'
(FORMAT PARQUET, OVERWRITE_OR_IGNORE TRUE)
""")

# Даже при нуле строк COPY создаст Parquet-файл с заданной схемой.
con.execute(f"""
COPY (SELECT * FROM quarantine_data)
TO '{escaped_quarantine_path}'
(FORMAT PARQUET, OVERWRITE_OR_IGNORE TRUE)
""")

print("Staging:", staging_path)
print("Строк в staging:", staging_count)
print()
print("Quarantine:", quarantine_path)
print("Строк в карантине:", quarantine_count)
print()
print("Строк в raw:", raw_count)
print("Баланс raw = staging + quarantine:",
      raw_count == staging_count + quarantine_count)

Staging: s3://staging/seoul_bike/source_ingested_on=2026-10-13/staging.parquet
Строк в staging: 8760

Quarantine: s3://staging/seoul_bike/source_ingested_on=2026-10-13/quarantine.parquet
Строк в карантине: 0

Строк в raw: 8760
Баланс raw = staging + quarantine: True


## Проверка сохранённых результатов

Повторно читаем staging.parquet и quarantine.parquet непосредственно из S3. Проверяем количество строк, схему карантина, отсутствие нарушений критических правил в staging и баланс исходных записей.

In [17]:
# Повторно читаем записанные Parquet-файлы непосредственно из S3.
staging_file = (
    f"read_parquet('{staging_path.replace(chr(39), chr(39) * 2)}')"
)
quarantine_file = (
    f"read_parquet('{quarantine_path.replace(chr(39), chr(39) * 2)}')"
)

# Сверяем число строк после записи и повторного чтения.
staging_read_count = con.execute(
    f"SELECT COUNT(*) FROM {staging_file}"
).fetchone()[0]

quarantine_read_count = con.execute(
    f"SELECT COUNT(*) FROM {quarantine_file}"
).fetchone()[0]

raw_read_count = con.execute(
    "SELECT COUNT(*) FROM raw_data"
).fetchone()[0]

print("Строк в raw:", raw_read_count)
print("Строк в прочитанном staging.parquet:", staging_read_count)
print("Строк в прочитанном quarantine.parquet:", quarantine_read_count)

print(
    "Баланс raw = staging + quarantine:",
    raw_read_count == staging_read_count + quarantine_read_count
)

# Проверяем схему карантина, даже если в нём нет записей.
quarantine_schema = con.execute(
    f"DESCRIBE SELECT * FROM {quarantine_file}"
).fetchall()

quarantine_columns = [column[0] for column in quarantine_schema]

print("\nПоля карантина:")
for column in quarantine_columns:
    print("-", column)

assert "quality_rule_ids" in quarantine_columns
assert "quarantine_reason" in quarantine_columns

# Повторно проверяем критические правила на прочитанном staging.
staging_critical_checks = con.execute(
    f"""
    SELECT
        COUNT(*) FILTER (
            WHERE observation_date IS NULL
               OR rented_bike_count IS NULL
        ) AS missing_required_fields,

        COUNT(*) FILTER (
            WHERE hour IS NULL OR hour < 0 OR hour > 23
        ) AS invalid_hours,

        COUNT(*) FILTER (
            WHERE rented_bike_count < 0
        ) AS negative_rentals,

        COUNT(*) FILTER (
            WHERE functioning_day = FALSE
              AND rented_bike_count <> 0
        ) AS inconsistent_functioning,

        COUNT(*) FILTER (
            WHERE quality_flag_humidity_out_of_range
        ) AS humidity_flags
    FROM {staging_file}
    """
).fetchone()

check_names = [
    "Пропущенные обязательные поля",
    "Недопустимые часы",
    "Отрицательные значения аренд",
    "Аренды при неработающей системе",
    "Флаги недопустимой влажности",
]

print("\nПроверки прочитанного staging:")
for name, value in zip(check_names, staging_critical_checks):
    print(f"{name}: {value}")

assert raw_read_count == staging_read_count + quarantine_read_count
assert all(value == 0 for value in staging_critical_checks[:4])

print("\nПроверка результатов завершена.")

Строк в raw: 8760
Строк в прочитанном staging.parquet: 8760
Строк в прочитанном quarantine.parquet: 0
Баланс raw = staging + quarantine: True

Поля карантина:
- Date
- Rented Bike Count
- Hour
- Temperature(°C)
- Humidity(%)
- Wind speed (m/s)
- Visibility (10m)
- Dew point temperature(°C)
- Solar Radiation (MJ/m2)
- Rainfall(mm)
- Snowfall (cm)
- Seasons
- Holiday
- Functioning Day
- ingested_on
- quality_rule_ids
- quarantine_reason
- source_ingested_on

Проверки прочитанного staging:
Пропущенные обязательные поля: 0
Недопустимые часы: 0
Отрицательные значения аренд: 0
Аренды при неработающей системе: 0
Флаги недопустимой влажности: 0

Проверка результатов завершена.


In [18]:
# Финальная проверка критических правил на Parquet, прочитанном из S3.
staging_validation = con.execute(
    f"""
    WITH checked AS (
        SELECT
            *,
            COUNT(*) OVER (
                PARTITION BY observation_date, hour
            ) AS key_count
        FROM {staging_file}
    )
    SELECT
        COUNT(*) FILTER (
            WHERE key_count > 1
        ) AS duplicate_key_rows,

        COUNT(*) FILTER (
            WHERE observation_date IS NULL
               OR rented_bike_count IS NULL
        ) AS missing_required_fields,

        COUNT(*) FILTER (
            WHERE hour IS NULL OR hour < 0 OR hour > 23
        ) AS invalid_hour_rows,

        COUNT(*) FILTER (
            WHERE rented_bike_count < 0
        ) AS negative_rental_rows,

        COUNT(*) FILTER (
            WHERE COALESCE(
                functioning_day = FALSE
                AND rented_bike_count <> 0,
                FALSE
            )
        ) AS inconsistent_functioning_rows,

        COUNT(*) FILTER (
            WHERE quality_flag_humidity_out_of_range
        ) AS humidity_flag_rows,

        COUNT(*) FILTER (
            WHERE quality_flag_humidity_out_of_range
                IS DISTINCT FROM (
                    humidity_pct IS NULL
                    OR humidity_pct < 0
                    OR humidity_pct > 100
                )
        ) AS humidity_flag_mismatch_rows

    FROM checked
    """
).fetchone()

validation_names = [
    "Повторяющиеся ключи Date + Hour",
    "Пропущенные обязательные поля",
    "Недопустимые часы",
    "Отрицательное количество аренд",
    "Аренды при неработающей системе",
    "Строки с установленным флагом влажности (справочно)",
    "Несоответствия флага фактической влажности",
]

print("Финальная проверка staging.parquet")

for name, count in zip(validation_names, staging_validation):
    print(f"{name}: {count}")

# Первые пять показателей должны быть равны нулю.
critical_validation = staging_validation[:5]

assert all(count == 0 for count in critical_validation), (
    "Обнаружены критические нарушения в staging.parquet."
)

# Установленные флаги допустимы, но они должны совпадать
# с фактическими значениями влажности.
humidity_flag_mismatch_rows = staging_validation[6]

assert humidity_flag_mismatch_rows == 0, (
    "Флаг влажности не соответствует фактическому значению."
)

print("\nВсе проверки выполнены успешно.")


Финальная проверка staging.parquet
Повторяющиеся ключи Date + Hour: 0
Пропущенные обязательные поля: 0
Недопустимые часы: 0
Отрицательное количество аренд: 0
Аренды при неработающей системе: 0
Строки с установленным флагом влажности (справочно): 0
Несоответствия флага фактической влажности: 0

Все проверки выполнены успешно.
